## From theory to practice

From Chapter 2 you should have a good grasp of NumPy, JAX, and PyTorch’s torch.tensor. This is all that is needed for this chapter, and nothing else is required. From the next chapter we will progress to their higher-level APIs.

I suggest a short exercise to let you train your first differentiable model from scratch:

1. Load a toy dataset: for example, one of those contained in scikit-learn datasets module.
2. Build a linear model (for regression or classification depending on the dataset). Think about how to make the code as modular as possible: as we will see, you will need at least two functions, one for initializing the parameters of the model and one for computing the model’s predictions.
4. Train the model via gradient descent. For now you can compute the gradients manually: try to imagine how you can make also this part modular, i.e., how do you change the gradient’s computation if you want to dynamically add or remove the bias from a model?
5. Plot the loss function and the accuracy on an independent test set. If you know some standard
machine learning, you can compare the results to other supervised learning models, such as a decision tree or a k-NN, always using scikit-learn.

In [1]:
!uv pip install matplotlib scikit-learn "numpy<2" --quiet
!uv pip install torch --quiet

## 1. Load dataset (regression)

In [2]:
from sklearn.datasets import fetch_california_housing
import torch

In [3]:
data = fetch_california_housing()

In [4]:
X = torch.from_numpy(data.data).float()
y = torch.from_numpy(data.target).float()

In [5]:
n = X.shape[0]  # size of the dataset
c = X.shape[1]  # features

print(f"Size of dataset n = {n}")
print(f"Number of features c = {c}")

Size of dataset n = 20640
Number of features c = 8


In [6]:
from sklearn.model_selection import train_test_split

# Split dataset 80% train / 20% test for step 4
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## 2. Least-squares regression model

When doing gradient descent, **we need to normalize our features**.

The analytical solution computes the minimizer directly in one shot, so feature scale only affects rounding error, not whether the answer is reached. Mathematically, scaling a feature just rescales the corresponding component of $w$; the optimum exists and is the same point regardless.

Gradient descent, by contrast, is an iterative process whose path to that optimum depends entirely on the geometry of the loss surface, and the loss surface's geometry is exactly what scaling changes: unscaled features make the Hessian $\frac{X^T X}{n}$ ill-conditioned, with the largest eigenvalue $\lambda_\max$ (steep direction) possibly thousands of times larger than the smallest $\lambda_\min$ (flat direction). Since a single learning rate $\eta$ must satisfy $\eta < 2 \lambda_\max$ for stability while progress along the flat directions is proportional to $\eta \lambda_\min$, bad conditioning forces you into a cruel tradeoff: small enough $\eta$ to avoid exploding in the steep direction, but then crawling forever along the flat one. Standardization compresses that eigenvalue spread (condition number drops from potentially huge to roughly the number of features), so one moderate learning rate works everywhere.

In [7]:
torch.linalg.cond((X_train.T @ X_train) / X_train.size(0))

tensor(1.2090e+08)

In [8]:
mean = X_train.mean(dim=0, keepdim=True)
std = X_train.std(dim=0, keepdim=True)

X_train = (X_train - mean) / (std + 1e-8)

In [9]:
torch.linalg.cond((X_train.T @ X_train) / X_train.size(0))

tensor(42.8292)

In [10]:
class RegressionModel:

    def __init__(self, c, _lambda=0.0):
        self.w = torch.normal(0, 0.01, size=(c + 1,))
        # We don't use b, we add 1 to the features w ~ (c + 1)
        # self.b = torch.zeros(1)
        self._lambda = _lambda  # l2 penalty

    def grad(self, X, y):
        X1 = torch.column_stack((X, torch.ones(X.size(0))))
        return X1.T @ (X1 @ self.w - y) / X.size(0) + self._lambda * self.w

    def forward(self, X):
        X1 = torch.column_stack((X, torch.ones(X.size(0))))
        return X1 @ self.w

In [11]:
def squared_loss(y, y_pred):
    return torch.mean((y - y_pred)**2)

## 3. Train

The book doesn't derive the gradient for the logistic regression. The softmax function makes it a bit complicated. So we use torch's autograd for gradient descent.

In [12]:
learning_rate = 0.1
epochs = 15000

In [13]:
model = RegressionModel(c)

In [14]:
for epoch in range(epochs):
    y_pred = model.forward(X_train)
    loss = squared_loss(y_train, y_pred)
    gradw = model.grad(X_train, y_train)
    model.w -= learning_rate * gradw
    # Check for convergence
    if torch.linalg.vector_norm(gradw) < 1e-5:
        print(f"Converged at epoch {epoch + 1}")
        break
    # print(f"Epoch {epoch + 1} - loss: {loss}")

Converged at epoch 1651


Compare against analytical solution:

In [15]:
X1_train = torch.column_stack((X_train, torch.ones(X_train.size(0))))
w_star = torch.linalg.inv(X1_train.T @ X1_train) @ X1_train.T @ y_train

In [16]:
torch.allclose(model.w, w_star, rtol=1e-2)

True

## 4. Validation

We'll do the same loop but save test accuracy.

In [17]:
X_test = (X_test - mean) / (std + 1e-8)

In [18]:
pred = model.forward(X_test)

In [19]:
squared_loss(y_test, pred)

tensor(0.5559)